In [0]:
CREATE OR REPLACE VIEW workspace.default.monthly_closure_audit_v2 AS
WITH monthly_volume AS (
  SELECT
    created_month,
    COUNT(*) AS resident_rat_report_count
  FROM workspace.default.rat_clean_v2
  WHERE resident_rat_report
  GROUP BY created_month
),

monthly_closed AS (
  SELECT
    created_month,

    COUNT(*) AS closed_resident_rat_report_count,

    MIN(closure_seconds)
      AS minimum_closure_seconds,

    PERCENTILE_CONT(0.5)
      WITHIN GROUP (ORDER BY closure_seconds)
      AS median_closure_seconds,

    MAX(closure_seconds)
      AS maximum_closure_seconds,

    ROUND(
      100.0 * COUNT_IF(closed_within_60_seconds) / COUNT(*),
      2
    ) AS percent_closed_within_60_seconds,

    ROUND(
      100.0 * COUNT_IF(
        DATE(created_date) = DATE(closed_date)
      ) / COUNT(*),
      2
    ) AS percent_recorded_closed_same_day

  FROM workspace.default.rat_clean_v2
  WHERE resident_rat_report
    AND closure_seconds IS NOT NULL
  GROUP BY created_month
)

SELECT
  v.created_month,
  v.resident_rat_report_count,
  COALESCE(c.closed_resident_rat_report_count, 0)
    AS closed_resident_rat_report_count,
  c.minimum_closure_seconds,
  c.median_closure_seconds,
  c.maximum_closure_seconds,
  c.percent_closed_within_60_seconds,
  c.percent_recorded_closed_same_day,

  CASE
    WHEN v.created_month < TIMESTAMP('2026-04-01')
      THEN 'Before observed transition'
    WHEN v.created_month = TIMESTAMP('2026-04-01')
      THEN 'Transition month'
    ELSE 'After observed transition'
  END AS observed_period

FROM monthly_volume v
LEFT JOIN monthly_closed c
  ON v.created_month = c.created_month;

CREATE OR REPLACE VIEW workspace.default.zip_coordinate_concentration_v2 AS
WITH coordinate_counts AS (
  SELECT
    zip_code,
    coordinate_key,
    COUNT(*) AS all_record_count,
    COUNT_IF(resident_rat_report)
      AS resident_rat_report_count,
    COUNT_IF(inspector_signs_record)
      AS inspector_signs_record_count
  FROM workspace.default.rat_clean_v2
  WHERE coordinate_key IS NOT NULL
  GROUP BY zip_code, coordinate_key
),

zip_totals AS (
  SELECT
    zip_code,
    SUM(all_record_count)
      AS geocoded_record_count,
    COUNT(*) AS distinct_coordinate_count,
    SUM(resident_rat_report_count)
      AS geocoded_resident_rat_report_count,
    COUNT_IF(resident_rat_report_count > 0)
      AS resident_rat_distinct_coordinate_count
  FROM coordinate_counts
  GROUP BY zip_code
),

all_coordinate_ranks AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY zip_code
      ORDER BY all_record_count DESC, coordinate_key
    ) AS coordinate_rank
  FROM coordinate_counts
),

resident_coordinate_ranks AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY zip_code
      ORDER BY resident_rat_report_count DESC, coordinate_key
    ) AS resident_coordinate_rank
  FROM coordinate_counts
  WHERE resident_rat_report_count > 0
),

top_all AS (
  SELECT
    zip_code,
    coordinate_key AS top_coordinate,
    all_record_count AS top_coordinate_record_count,
    inspector_signs_record_count
      AS top_coordinate_inspector_signs_count
  FROM all_coordinate_ranks
  WHERE coordinate_rank = 1
),

top_resident AS (
  SELECT
    zip_code,
    coordinate_key AS top_resident_coordinate,
    resident_rat_report_count
      AS top_coordinate_resident_rat_report_count
  FROM resident_coordinate_ranks
  WHERE resident_coordinate_rank = 1
)

SELECT
  z.zip_code,
  z.geocoded_record_count,
  z.distinct_coordinate_count,
  a.top_coordinate,
  a.top_coordinate_record_count,

  ROUND(
    100.0 * a.top_coordinate_record_count
      / NULLIF(z.geocoded_record_count, 0),
    2
  ) AS top_coordinate_share_percent,

  a.top_coordinate_inspector_signs_count,

  z.geocoded_resident_rat_report_count,
  z.resident_rat_distinct_coordinate_count,
  r.top_resident_coordinate,
  r.top_coordinate_resident_rat_report_count,

  ROUND(
    100.0 * r.top_coordinate_resident_rat_report_count
      / NULLIF(z.geocoded_resident_rat_report_count, 0),
    2
  ) AS top_resident_coordinate_share_percent,

  CASE
    WHEN z.geocoded_record_count >= 20
      AND (
        100.0 * a.top_coordinate_record_count
          / NULLIF(z.geocoded_record_count, 0)
      ) >= 50
    THEN 'High coordinate concentration'
    ELSE 'No high concentration flag'
  END AS coordinate_concentration_status

FROM zip_totals z
LEFT JOIN top_all a
  ON z.zip_code = a.zip_code
LEFT JOIN top_resident r
  ON z.zip_code = r.zip_code;

SELECT *
FROM workspace.default.zip_coordinate_concentration_v2
WHERE zip_code IN ('10035', '11379', '11415', '11423', '11430')
ORDER BY zip_code;

SELECT
  COUNT_IF(closure_seconds < 0) AS negative_closure_records,
  MIN(closure_seconds) AS minimum_closure_seconds,
  MAX(closure_seconds) AS maximum_closure_seconds
FROM workspace.default.rat_clean_v2
WHERE resident_rat_report
  AND closure_seconds IS NOT NULL;